# Advanced 03 — Modes of Convergence and Weak Convergence

Practice notebook for the **"Modes of Convergence and Weak Convergence"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Almost Sure Convergence

A sequence of random variables \(X_n\) converges **almost surely** to
\(X\), written \(X_n \to} X\), if

$$
P\!\left(\{\omega : X_n(\omega) \to X(\omega)\}\right) = 1.
$$

This is a *pathwise* statement: for almost every realization \(\omega\),
the sequence of real numbers \(X_n(\omega)\) converges to \(X(\omega)\)
in the ordinary calculus sense. The canonical example is the Strong Law of
Large Numbers: for i.i.d. integrable \(X_n\),

$$
\bar{X}_n = \frac{1}{n}\sum_{k=1}^n X_k \to} E[X_1].
$$

Below we draw a *single* sequence of running means \(\bar{X}_n\) for
i.i.d. \(\mathrm{Normal}(\mu,\sigma^2)\) variables and watch the one
realized path converge to \(\mu\). We also overlay a few extra paths to
show that *each* path individually settles down — that is the content of
"almost sure".

**Your turn:** increase `n` to 100000 and watch the running mean tighten
around \(\mu\); then replace the Normal by a Rademacher \(\pm 1\) and
confirm a.s. convergence still holds (only the first moment matters).


In [ ]:
# Almost sure convergence: a single sequence of running means -> mu
rng = np.random.default_rng(7)

n = 5000
mu_true = 1.0
sigma_true = 2.0

# One "main" path we follow in detail, plus a few background paths
main_path = rng.normal(loc=mu_true, scale=sigma_true, size=n)
bar_main = np.cumsum(main_path) / np.arange(1, n + 1)

bg_paths = rng.normal(loc=mu_true, scale=sigma_true, size=(8, n))
bar_bg = np.cumsum(bg_paths, axis=1) / np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(9, 5))
for j in range(8):
    ax.plot(bar_bg[j], color="lightgray", alpha=0.6, lw=0.7)
ax.plot(bar_main, color="steelblue", lw=1.4, label="$\\bar{X}_n(\\omega)$ (one path)")
ax.axhline(mu_true, color="crimson", lw=2, label=f"$\\mu={mu_true}$")
ax.set_title("Almost sure convergence: $\\bar{X}_n \\to \\mu$ (one realized path)")
ax.set_xlabel("$n$")
ax.set_ylabel("$\\bar{X}_n$")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final value of the realized path: bar_X_n = {bar_main[-1]:.5f}")
print(f"Distance to mu at n={n}:          |bar_X_n - mu| = {abs(bar_main[-1] - mu_true):.5f}")


---
## Part 2 — Convergence in Probability

\(X_n \to X\) if for every \(\varepsilon>0\),

$$
\lim_{n\to\infty} P\big(|X_n - X| > \varepsilon\big) = 0.
$$

This is weaker than a.s. convergence: it only asks that the *probability* of a
deviation of size \(\varepsilon\) vanishes, not that *every* path eventually
stays close. Both a.s. convergence and \(L^p\) convergence imply convergence
in probability (the converses generally fail).

We cannot observe \(P(|X_n-X|>\varepsilon)\) directly, but we can estimate it
by **Monte Carlo**: simulate many independent copies of \(X_n-X\) and count
how often \(|X_n-X|>\varepsilon\). We use the running mean again,
\(X_n=\bar{X}_n\), \(X=\mu\), so \(X_n-X=\bar{X}_n-\mu\), and estimate

$$
\widehat{p}_n(\varepsilon) = \frac{1}{M}\sum_{m=1}^M
\mathbf{1}\{|\bar{X}_n^{(m)}-\mu|>\varepsilon\}.
$$

For finite \(n\) the SLLN gives \(\bar{X}_n-\mu \sim
\mathrm{Normal}(0, \sigma^2/n)\), so we can also compare with the exact
probability \(2\,(1-\Phi(\varepsilon\sqrt{n}/\sigma))\).

**Your turn:** shrink \(\varepsilon\) and watch the convergence slow down
(larger \(n\) is needed); change \(\sigma\) and explain the direction of
the change in the curves.


In [ ]:
# Convergence in probability: estimate P(|Xn - X| > eps) across replicates
rng = np.random.default_rng(123)

M = 4000           # number of independent replicates of Xn - X
ns = np.array([10, 25, 50, 100, 250, 500, 1000, 2500, 5000])
mu = 0.0
sigma = 1.0
eps = 0.1

p_hat = []
p_exact = []
for n in ns:
    # M replicates of bar_X_n - mu, each based on n i.i.d. draws
    X = rng.normal(loc=mu, scale=sigma, size=(M, n))
    bar_X = X.mean(axis=1) - mu
    p_hat.append(np.mean(np.abs(bar_X) > eps))
    # Exact: 2*(1 - Phi(eps * sqrt(n) / sigma))
    p_exact.append(2.0 * (1.0 - stats.norm.cdf(eps * np.sqrt(n) / sigma)))

p_hat = np.array(p_hat)
p_exact = np.array(p_exact)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ns, p_hat, "o-", color="steelblue", lw=1.5, label=f"$\\widehat{{p}}_n(\\varepsilon={eps})$ (Monte Carlo)")
ax.plot(ns, p_exact, "--", color="crimson", lw=1.5, label=f"exact $2(1-\\Phi(\\varepsilon\\sqrt{{n}}/\\sigma))$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title(f"Convergence in probability: $P(|\\bar{{X}}_n-\\mu|>\\varepsilon={eps})\\to 0$")
ax.set_xlabel("$n$")
ax.set_ylabel("$P(|\\bar{X}_n-\\mu| > \\varepsilon)$")
ax.legend()
plt.tight_layout()
plt.show()

for n_i, ph, pe in zip(ns, p_hat, p_exact):
    print(f"n={n_i:5d}: MC p_hat={ph:.4f}  exact={pe:.4e}")


---
## Part 3 — Convergence in \(L^p\)

\(X_n \to X\) in \(L^p\), \(1\leq p<\infty\), if

$$
\lim_{n\to\infty} E\big[|X_n - X|^p\big] = 0.
$$

This is an *averaged* statement (an \(L^p\)-norm statement) and it implies
convergence in probability (Markov's inequality:
\(P(|X_n-X|>\varepsilon)\leq E[|X_n-X|^p]/\varepsilon^p\)). It does **not**
generally imply a.s. convergence, and a.s. convergence does not generally
imply \(L^p\) convergence (a counterexample is sketched in the problem set).

Again with \(X_n=\bar{X}_n\), \(X=\mu\) and \(X_n\) i.i.d. with finite
\(p\)-th moment, \(E[|\bar{X}_n-\mu|^p]\to 0\). For \(p=2\) this is
just \(\operatorname{Var}(\bar{X}_n)=\sigma^2/n\to 0\). We estimate
\(E[|\bar{X}_n-\mu|^p]\) by Monte Carlo for \(p\in\{1,2,4\}\) and
check the rates \(n^{-p/2}\) (the CLT scale).

**Your turn:** add \(p=8\) and confirm the rate \(n^{-4}\); then try
\(X_n\) heavy-tailed (Student-t with \(\nu=3\)) and watch the \(p=4\)
moment blow up — finite \(p\)-th moment is required for \(L^p\)
convergence.


In [ ]:
# Lp convergence: Monte Carlo estimate of E[|Xn - X|^p] for several p
rng = np.random.default_rng(31)

M = 20000
ns = np.array([10, 25, 50, 100, 250, 500, 1000, 2500, 5000])
mu = 0.0
sigma = 1.0
ps = [1, 2, 4]
colors = {1: "steelblue", 2: "crimson", 4: "darkgreen"}

fig, ax = plt.subplots(figsize=(9, 5))
for p in ps:
    moments = []
    for n in ns:
        X = rng.normal(loc=mu, scale=sigma, size=(M, n))
        bar_X = X.mean(axis=1) - mu
        moments.append(np.mean(np.abs(bar_X) ** p))
    moments = np.array(moments)
    ax.plot(ns, moments, "o-", color=colors[p], lw=1.5,
            label=f"$E[|\\bar{{X}}_n-\\mu|^{{ {p} }}]$ (MC)")
    # Reference rate n^{-p/2}: pin the curve to the first point
    ref = moments[0] * (ns / ns[0]) ** (-p / 2.0)
    ax.plot(ns, ref, "--", color=colors[p], lw=1.0, alpha=0.7,
            label=f"$C\\,n^{{ {-p/2:.1f} }}$ reference")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("$L^p$ convergence: $E[|\\bar{X}_n-\\mu|^p]\\to 0$ at rate $n^{-p/2}$")
ax.set_xlabel("$n$")
ax.set_ylabel("$E[|\\bar{X}_n-\\mu|^p]$")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for p in ps:
    m_last = []
    for n in ns[-3:]:
        X = rng.normal(loc=mu, scale=sigma, size=(M, n))
        bar_X = X.mean(axis=1) - mu
        m_last.append(np.mean(np.abs(bar_X) ** p))
    print(f"p={p}: E[|bar_X_n-mu|^p] for n={ns[-3:].tolist()} = "
          f"{[f'{v:.4e}' for v in m_last]}  (rate ~ n^{{ {-p/2:.0f} }})")


---
## Part 4 — Weak Convergence of Distributions

A sequence \(X_n\) converges **in distribution** (or **weakly**) to
\(X\), written \(X_n \Rightarrow X\), if

$$
\lim_{n\to\infty} F_{X_n}(x) = F_X(x)
$$

for every \(x\) at which \(F_X\) is continuous. Weak convergence is about
*distributions*, not sample paths — it is the right notion for CLT-type
results and asymptotic distributions of estimators.

A useful equivalent characterization (Portmanteau) is
\(E[f(X_n)]\to E[f(X)]\) for every bounded continuous \(f\). We
illustrate both faces below:

1. **Empirical CDFs**: let \(X_n\sim\mathrm{Normal}(0, 1/n)\) (so
   \(X_n\Rightarrow 0\)). We plot the empirical CDF of samples of
   \(X_n\) for increasing \(n\) and watch it collapse to the step
   function \(x\mapsto \mathbf{1}\{x\geq 0\}\), the CDF of the point
   mass at 0.
2. **Portmanteau with bounded continuous \(f\)**: take
   \(f=\arctan\) (bounded by \(\pi/2\)) and compute \(E[f(X_n)]\)
   by Monte Carlo; it should converge to \(\arctan(0)=0\).

Note: the same \(X_n\Rightarrow 0\) does **not** give \(L^p\) or a.s.
convergence here unless the variables are coupled appropriately — weak
convergence is genuinely weaker.

**Your turn:** replace the limit by \(X\sim\mathrm{Normal}(0,1)\) by
taking \(X_n = Z + 1/n\) with \(Z\sim\mathrm{Normal}(0,1)\) (so
\(X_n\Rightarrow Z\)); confirm the empirical CDFs line up with
\(\Phi\) and \(E[\arctan(X_n)]\to E[\arctan(Z)]\).


In [ ]:
# Weak convergence: empirical CDFs collapsing to a point mass + Portmanteau with arctan
rng = np.random.default_rng(5)

M = 4000                       # samples per n
ns = np.array([1, 5, 20, 100, 500])
xs = np.linspace(-1.5, 1.5, 800)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (1) Empirical CDFs of X_n ~ Normal(0, 1/n) approaching delta_0
for n in ns:
    samples = rng.normal(0.0, 1.0 / np.sqrt(n), size=M)
    ecdf = np.array([np.mean(samples <= x) for x in xs])
    axes[0].plot(xs, ecdf, label=f"$n={n}$", lw=1.2)
axes[0].plot(xs, (xs >= 0).astype(float), color="black", lw=2, linestyle="--",
             label=r"$F_{\\delta_0}(x)=\\mathbf{1}\{x\\geq 0\}$")
axes[0].set_title("Empirical CDFs: $X_n\\sim N(0,1/n)\\Rightarrow \\delta_0$")
axes[0].set_xlabel("$x$")
axes[0].set_ylabel("$F_{X_n}(x)$")
axes[0].legend(fontsize=8)

# (2) Portmanteau: E[arctan(X_n)] -> arctan(0) = 0
ns_dense = np.unique(np.geomspace(1, 2000, 25).astype(int))
Ef = []
for n in ns_dense:
    samples = rng.normal(0.0, 1.0 / np.sqrt(n), size=M)
    Ef.append(np.mean(np.arctan(samples)))
Ef = np.array(Ef)

axes[1].plot(ns_dense, Ef, "o-", color="steelblue", lw=1.4,
            label="$E[\\arctan(X_n)]$ (MC)")
axes[1].axhline(np.arctan(0.0), color="crimson", lw=2, linestyle="--",
               label="$\\arctan(0)=0$")
axes[1].set_xscale("log")
axes[1].set_title("Portmanteau: $E[f(X_n)]\\to E[f(X)]$ for $f=\\arctan$")
axes[1].set_xlabel("$n$")
axes[1].set_ylabel("$E[\\arctan(X_n)]$")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Last E[arctan(X_n)] (n={ns_dense[-1]}): {Ef[-1]:.5e} (target 0)")
print(f"Max |E[arctan(X_n)] - 0| over all n:    {np.max(np.abs(Ef)):.5e}")


---
## Part 5 — Continuous Mapping Theorem and Delta Method

**Continuous Mapping Theorem.** If \(X_n \Rightarrow X\) and
\(g\) is continuous, then \(g(X_n) \Rightarrow g(X)\).

**Delta method.** If \(\sqrt{n}(T_n-\theta) \Rightarrow N(0,\tau^2)\)
and \(g\) is differentiable at \(\theta\) with
\(g'(\theta)\ne 0\), then

$$
\sqrt{n}\big(g(T_n)-g(\theta)\big) \Rightarrow
N\!\big(0,(g'(\theta))^2\tau^2\big).
$$

The intuition: small perturbations of \(T_n\) around \(\theta\) are
linearly amplified by \(g'(\theta)\) in the limit.

We illustrate this with \(T_n=\bar{X}_n\) for i.i.d.
\(\mathrm{Normal}(\mu,\sigma^2)\), so
\(\sqrt{n}(\bar{X}_n-\mu)\Rightarrow N(0,\sigma^2)\). Take
\(g(x)=x^2\), so \(g'(\mu)=2\mu\). The delta method predicts

$$
\sqrt{n}\big(\bar{X}_n^2 - \mu^2\big) \Rightarrow
N\!\big(0, 4\mu^2\sigma^2\big).
$$

We Monte-Carlo the standardized statistic
\(\sqrt{n}(\bar{X}_n^2-\mu^2)\) and compare its histogram with the
predicted \(N(0, 4\mu^2\sigma^2)\). As a bonus, the CMT says
\(\bar{X}_n^2 \Rightarrow \mu^2\), which we verify via the empirical CDF.

**Your turn:** pick \(\mu=0\) and explain why the \(N(0,4\mu^2\sigma^2)\)
prediction degenerates (\(g'(0)=0\)); then use a second-order delta method
with \(g''(0)=2\) to get the right limit \(N(\sigma^2/\sqrt{2}, \dots)\).


In [ ]:
# Delta method: sqrt(n) (g(bar_X_n) - g(mu)) -> N(0, (g'(mu))^2 sigma^2)  with g(x)=x^2
rng = np.random.default_rng(99)

M = 40000          # Monte Carlo replicates
n = 400
mu = 1.5
sigma = 1.0

# Draw M replicates of bar_X_n
X = rng.normal(loc=mu, scale=sigma, size=(M, n))
bar_X = X.mean(axis=1)

# Delta-method statistic: sqrt(n) * (g(bar_X_n) - g(mu)) with g(x) = x^2
g = lambda x: x ** 2
gprime_mu = 2.0 * mu
stat = np.sqrt(n) * (g(bar_X) - g(mu))
limit_var = (gprime_mu ** 2) * sigma ** 2
limit_sd = np.sqrt(limit_var)

# Also the CMT target: g(X) = mu^2, i.e. bar_X_n^2 => mu^2
g_target = g(mu)

xs = np.linspace(-6 * limit_sd, 6 * limit_sd, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: delta-method statistic vs predicted Gaussian limit
axes[0].hist(stat, bins=80, density=True, color="steelblue", alpha=0.6,
             label=f"$\\sqrt{{n}}(\\bar{{X}}_n^2-\\mu^2)$, n={n}")
axes[0].plot(xs, stats.norm.pdf(xs, 0, limit_sd), color="crimson", lw=2,
             label=f"$N(0,\\,4\\mu^2\\sigma^2={limit_var:.2f})$")
axes[0].set_title("Delta method: $\\sqrt{n}(g(\\bar{X}_n)-g(\\mu))\\Rightarrow N(0,(g')^2\\sigma^2)$")
axes[0].set_xlabel("statistic")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=9)

# Right: CMT — bar_X_n^2 (as a random variable) concentrates near mu^2
axes[1].hist(g(bar_X), bins=80, density=True, color="lightgray", alpha=0.7,
             label="$\\bar{X}_n^2$")
axes[1].axvline(g_target, color="crimson", lw=2, label=f"$g(\\mu)=\\mu^2={g_target:.2f}$")
axes[1].set_title(f"Continuous mapping: $\\bar{{X}}_n^2 \\Rightarrow \\mu^2={g_target:.2f}$")
axes[1].set_xlabel("$\\bar{X}_n^2$")
axes[1].set_ylabel("density")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

ks_stat, ks_p = stats.kstest((stat - stat.mean()) / stat.std(), "norm")
print(f"Delta-method stat: mean={stat.mean():.4f}, var={stat.var():.4f}  (predicted var={limit_var:.4f})")
print(f"KS (standardized stat vs N(0,1)): stat={ks_stat:.4f}, p={ks_p:.3f}")
print(f"CMT: mean of bar_X_n^2 = {g(bar_X).mean():.4f} (target mu^2 = {g_target:.4f})")


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the definitions of almost sure, in probability, and \(L^p\)
    convergence, and the two implications
    \(X_n\toX \Rightarrow X_n\toX\) and
    \(X_n\to}X \Rightarrow X_n\toX\).
    Explain in one sentence why the converses generally fail.

2. Construct an explicit example showing that convergence in probability
    does **not** imply almost sure convergence. (Hint: independent blocks that
    are 1 with probability \(1/n\) and 0 otherwise, arranged so every index
    is hit infinitely often.) Sketch the intuition.

3. Using the Part 3 code with \(p=2\), verify numerically that
    \(E[|\bar{X}_n-\mu|^2]=\sigma^2/n\) for
    \(X_n\sim\mathrm{Normal}(\mu,\sigma^2)\). Why does \(L^2\)
    convergence imply convergence in probability via Markov's inequality?

4. Define weak convergence \(X_n\Rightarrow X\). Give an example where
    \(X_n\Rightarrow X\) but \(X_n\) does **not** converge to \(X\) in
    probability. What feature of the joint coupling is responsible?

5. State the delta method. If \(\sqrt{n}(T_n-\theta)\Rightarrow
    N(0,\tau^2)\) and \(g(x)=\exp(x)\), what is the asymptotic
    distribution of \(\sqrt{n}(g(T_n)-g(\theta))\)? Verify the variance
    formula using the Part 5 code by setting \(g(x)=\exp(x)\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Definitions:

- **a.s.:** \(P(\{\omega:X_n(\omega)\to X(\omega)\})=1\).
- **in probability:** \(\forall\varepsilon>0,\;
  P(|X_n-X|>\varepsilon)\to 0\).
- **in \(L^p\):** \(E[|X_n-X|^p]\to 0\) for \(1\leq p<\infty\).

Implications: \(L^p\Rightarrow P\) by Markov,
\(P(|X_n-X|>\varepsilon)\leq E[|X_n-X|^p]/\varepsilon^p\to 0\); and
a.s. \(\Rightarrow P\) because if the event
\(A=\{X_n\to X\}\) has probability 1, then on \(A\) the indicators
\(\mathbf{1}\{|X_n-X|>\varepsilon\}\) are eventually 0, so by dominated
convergence \(P(|X_n-X|>\varepsilon)=E[\mathbf{1}\{|X_n-X|>\varepsilon\}]\to 0\).

The converses fail because a.s. convergence is a *pathwise* requirement
(\(P\)-convergence only controls marginal deviations, not joint behavior
across \(n\)), and \(L^p\) controls the *size* of deviations on average,
not just their probability.

**2.** Partition \(\mathbb{N}\) into blocks
\(I_k=\{2^k,\dots,2^{k+1}-1\}\) (length \(2^k\)). On block \(I_k\)
let \(X_n\) be independent with \(P(X_n=1)=1/k\) and \(P(X_n=0)=1-1/k\),
and let \(X=0\). For fixed \(\varepsilon\in(0,1)\),
\(P(|X_n-X|>\varepsilon)=1/k\to 0\) on block \(I_k\), so
\(X_n\to0\). But
\(\sum_n P(X_n=1)=\sum_k |I_k|\cdot(1/k)=\sum_k 2^k/k=\infty\); since the
\(X_n\) are independent, the Borel–Cantelli lemma (second part) gives
\(P(X_n=1\text{ i.o.})=1\). Hence \(X_n(\omega)\not\to 0\) for almost
every \(\omega\): \(X_n\) does not converge a.s. The issue is that the
probabilities vanish, but new "spikes" keep appearing at later \(n\).

**3.** With \(X_n\sim\mathrm{Normal}(\mu,\sigma^2)\) i.i.d. and
\(\bar{X}_n\) their mean,
\(\bar{X}_n-\mu\sim\mathrm{Normal}(0,\sigma^2/n)\), so

$$
E[|\bar{X}_n-\mu|^2]=\operatorname{Var}(\bar{X}_n)=\sigma^2/n.
$$

Running the Part 3 code with \(p=2\) prints values scaling like
\(1/n\), e.g. `~1.0e-2` at \(n=100\) for \(\sigma=1\). Markov's
inequality gives \(P(|\bar{X}_n-\mu|>\varepsilon)\leq
E[|\bar{X}_n-\mu|^2]/\varepsilon^2=\sigma^2/(n\varepsilon^2)\to 0\), so
\(L^2\Rightarrow P\).

**4.** Weak convergence only constrains the *marginal* distributions
\(F_{X_n}\to F_X\); it says nothing about how \(X_n\) and \(X\) are
coupled on a common probability space. Take
\(X_n=X\sim\mathrm{Normal}(0,1)\) for all \(n\), but with
\(X_n=-X\) (perfect anticorrelation). Then
\(X_n\Rightarrow X\) trivially (same distribution), yet
\(|X_n-X|=2|X|\) is a fixed non-degenerate random variable, so
\(P(|X_n-X|>\varepsilon)\not\to 0\). Convergence in probability depends on
the *joint law* \((X_n,X)\), which weak convergence does not control.

**5.** With \(g(x)=\exp(x)\), \(g'(\theta)=\exp(\theta)\), so the
delta method gives

$$
\sqrt{n}(e^{T_n}-e^{\theta})\Rightarrow N\!\big(0,\,
e^{2\theta}\tau^2\big).
$$

To verify in Part 5, replace `g = lambda x: x ** 2` by `g = lambda x: np.exp(x)`
and set `gprime_mu = np.exp(mu)`. With \(\mu=1.5\), \(\sigma=1\),
the predicted limit variance is \(e^{2\mu}\sigma^2=e^{3}\approx 20.09\).
The printed `var` of the delta-method statistic will match this value, and the
KS p-value against \(N(0, e^{2\mu}\sigma^2)\) will be large.

</details>
